# Unsupervised Learning: Trade & Ahead


## Problem Statement


### Context

The stock market has consistently proven to be a good place to invest in and save for the future. There are a lot of compelling reasons to invest in stocks. It can help in fighting inflation, create wealth, and also provides some tax benefits. Good steady returns on investments over a long period of time can also grow a lot more than seems possible. Also, thanks to the power of compound interest, the earlier one starts investing, the larger the corpus one can have for retirement. Overall, investing in stocks can help meet life's financial aspirations.

It is important to maintain a diversified portfolio when investing in stocks in order to maximise earnings under any market condition. Having a diversified portfolio tends to yield higher returns and face lower risk by tempering potential losses when the market is down. It is often easy to get lost in a sea of financial metrics to analyze while determining the worth of a stock, and doing the same for a multitude of stocks to identify the right picks for an individual can be a tedious task. By doing a cluster analysis, one can identify stocks that exhibit similar characteristics and ones which exhibit minimum correlation. This will help investors better analyze stocks across different market segments and help protect against risks that could make the portfolio vulnerable to losses.

### Objective

Trade&Ahead is a financial consultancy firm who provide their customers with personalized investment strategies. They have hired you as a Data Scientist and provided you with data comprising stock price and some financial indicators for a few companies listed under the New York Stock Exchange. They have assigned you the tasks of analyzing the data, grouping the stocks based on the attributes provided, and sharing insights about the characteristics of each group.

### Data Dictionary

* Ticker Symbol: An abbreviation used to uniquely identify publicly traded shares of a particular stock on a particular stock market
* Company: Name of the company
* GICS Sector: The specific economic sector assigned to a company by the Global Industry Classification Standard (GICS) that best defines its business operations
* GICS Sub Industry: The specific sub-industry group assigned to a company by the Global Industry Classification Standard (GICS) that best defines its business operations
* Current Price: Current stock price in dollars
* Price Change: Percentage change in the stock price in 13 weeks
* Volatility: Standard deviation of the stock price over the past 13 weeks
* ROE: A measure of financial performance calculated by dividing net income by shareholders' equity (shareholders' equity is equal to a company's assets minus its debt)
* Cash Ratio: The ratio of a company's total reserves of cash and cash equivalents to its total current liabilities
* Net Cash Flow: The difference between a company's cash inflows and outflows (in dollars)
* Net Income: Revenues minus expenses, interest, and taxes (in dollars)
* Earnings Per Share: Company's net profit divided by the number of common shares it has outstanding (in dollars)
* Estimated Shares Outstanding: Company's stock currently held by all its shareholders
* P/E Ratio: Ratio of the company's current stock price to the earnings per share
* P/B Ratio: Ratio of the company's stock price per share by its book value per share (book value of a company is the net difference between that company's total assets and total liabilities)


#### End-to-end process flow


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

steps = [
    ("Define Business Problem\n(cluster 340 NYSE stocks by financial profile)", "#808080"),
    ("Load & Inspect Data\n(stock_data.csv: 340 rows, 15 columns)", "#808080"),
    ("Data Overview & Quality Checks\n(shape, dtypes, duplicates, missing values)", "#808080"),
    ("Exploratory Data Analysis\n(univariate distributions plus 6 leading questions)", "#2E86C1"),
    ("Data Preprocessing\n(review outliers, standardize features)", "#2E86C1"),
    ("K-Means Clustering\n(elbow and silhouette scores select k=4, profile clusters)", "#2E7D32"),
    ("Hierarchical Clustering\n(compare linkages/distances, choose Ward at k=4, profile clusters)", "#7D3C98"),
    ("Compare K-Means vs Hierarchical\n(cross-tab agreement, runtime)", "#CC5500"),
    ("Actionable Insights & Recommendations\n(segment-level guidance for advisors)", "#CC5500"),
]

box_h = 0.85
gap = 0.35
stack_h = len(steps) * (box_h + gap)
title_y = stack_h + 2.5
legend_top_y = stack_h + 1.7

fig, ax = plt.subplots(figsize=(9, 14.5))

y = stack_h
centers = []
for label, color in steps:
    y -= (box_h + gap)
    box = FancyBboxPatch((0.5, y), 8, box_h, boxstyle="round,pad=0.08,rounding_size=0.12",
                          linewidth=1.2, edgecolor=color, facecolor=color, alpha=0.18)
    ax.add_patch(box)
    ax.text(4.5, y + box_h / 2, label, ha="center", va="center", fontsize=9, color="#222222")
    centers.append(y + box_h / 2)

for i in range(len(centers) - 1):
    ax.annotate("", xy=(4.5, centers[i + 1] + box_h / 2 + 0.03), xytext=(4.5, centers[i] - box_h / 2 - 0.03),
                arrowprops=dict(arrowstyle="-|>", color="#555555", linewidth=1.3))

# Title, drawn directly in data coordinates so it cannot collide with the legend below it
ax.text(4.5, title_y, "Trade & Ahead: End-to-End Clustering Workflow",
        ha="center", va="center", fontsize=13, fontweight="bold", color="#111111")

# Legend, also drawn manually in data coordinates (a vertical list of color swatches + labels)
legend_entries = [
    ("#808080", "Setup & Data Understanding"),
    ("#2E86C1", "EDA & Preprocessing"),
    ("#2E7D32", "K-Means Clustering"),
    ("#7D3C98", "Hierarchical Clustering"),
    ("#CC5500", "Comparison & Recommendations"),
]
row_h = 0.32
for i, (color, label) in enumerate(legend_entries):
    ly = legend_top_y - i * row_h
    swatch = FancyBboxPatch((2.6, ly - 0.11), 0.3, 0.22, boxstyle="round,pad=0.02,rounding_size=0.03",
                             linewidth=1.0, edgecolor=color, facecolor=color, alpha=0.35)
    ax.add_patch(swatch)
    ax.text(3.05, ly, label, ha="left", va="center", fontsize=8.5, color="#333333")

ax.set_xlim(0, 9)
ax.set_ylim(0, title_y + 0.9)
ax.axis("off")
plt.tight_layout()
plt.show()


The diagram above summarizes the path taken through this notebook, from the initial business problem through to the final investment recommendations: data understanding and quality checks, exploratory analysis, preprocessing, the two clustering approaches run in parallel (K-means and hierarchical), a direct comparison of the two solutions, and the resulting actionable insights.


## Importing necessary libraries and data


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

from yellowbrick.cluster import SilhouetteVisualizer

from scipy.cluster.hierarchy import dendrogram, linkage, cophenet, fcluster
from scipy.spatial.distance import pdist

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [ ]:
# Load the raw stock/financial-indicator dataset
df = pd.read_csv("stock_data.csv")
df.head()


## Data Overview


* Observations
* Sanity checks


In [ ]:
# Basic shape and dtype/non-null overview
print("Shape:", df.shape)
df.info()


In [ ]:
# Duplicate check
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate tickers:", df.duplicated(subset=["Ticker Symbol"]).sum())

# Missing value check
df.isnull().sum()


In [ ]:
# Summary statistics for every column (numeric + categorical)
df.describe(include="all").T


**Observations:**

* The dataset has 340 companies and 15 columns, with no missing values and no duplicate rows or tickers.
* 11 of the columns are numeric financial indicators; the remaining 4 are identifiers (Ticker Symbol, Security) and categorical groupings (GICS Sector, GICS Sub Industry).
* Several numeric columns (Net Cash Flow, Net Income, Estimated Shares Outstanding) span several orders of magnitude, reflecting the wide range of company sizes in the dataset (from small caps to mega caps). These will need to be scaled before clustering.
* P/B Ratio has a negative minimum, which is possible when a company's liabilities exceed its assets (negative book value); this is not a data error.


## Exploratory Data Analysis (EDA)


* EDA is an important part of any project involving data.
* It is important to investigate and understand the data better before building a model with it.
* A few questions have been mentioned below which will help you approach the analysis in the right manner and generate insights from the data.
* A thorough analysis of the data, in addition to the questions mentioned below, should be done.


**Questions**:

1. What does the distribution of stock prices look like?
2. The stocks of which economic sector have seen the maximum price increase on average?
3. How are the different variables correlated with each other?
4. Cash ratio provides a measure of a company's ability to cover its short-term obligations using only cash and cash equivalents. How does the average cash ratio vary across economic sectors?
5. P/E ratios can help determine the relative value of a company's shares as they signify the amount of money an investor is willing to invest in a single share of a company per dollar of its earnings. How does the P/E ratio vary, on average, across economic sectors?
6. How does the volatility in stock prices vary across economic sectors?


### Univariate Analysis


Before answering the Leading Questions above (which mostly involve comparisons *across* groups), this section looks at the distribution of every individual variable on its own: both the numeric financial indicators and the two categorical groupings.


In [ ]:
# Distribution of every numeric feature
num_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(4, 3, figsize=(16, 16))
axes = axes.flatten()
for ax, col in zip(axes, num_cols_all):
    sns.histplot(df[col], bins=25, kde=True, color="#4C72B0", ax=ax)
    ax.set_title(col, fontsize=10)
for ax in axes[len(num_cols_all):]:
    ax.axis("off")
fig.suptitle("Distribution of All Numeric Features", fontsize=14)
plt.tight_layout()
plt.show()


**Insight:** Most of the size/profitability-related features are heavily right-skewed: Current Price (skew 7.1), P/E Ratio (6.7), ROE (5.8), Cash Ratio (4.5), Net Cash Flow and Estimated Shares Outstanding (3.9 each) all have a long tail of a few extreme companies pulling the mean well above the median. Price Change is closer to symmetric with a mild left skew, and Earnings Per Share is left-skewed (a handful of companies with unusually large losses). This confirms, ahead of clustering, that every numeric feature will need to be standardized so the handful of extreme-scale companies (e.g., Net Income in the billions) don't dominate the distance calculation.


In [ ]:
# Number of companies per GICS Sector
sector_counts_all = df["GICS Sector"].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("Blues_r", len(sector_counts_all))
ax.barh(sector_counts_all.index[::-1], sector_counts_all.values[::-1], color=colors[::-1])
ax.set_xlabel("Number of Companies")
ax.set_title("Number of Companies by GICS Sector")
for i, v in enumerate(sector_counts_all.values[::-1]):
    ax.text(v + 2, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()

sector_counts_all


**Insight:** Industrials (53), Financials (49), Health Care (40), and Consumer Discretionary (40) are the most heavily represented sectors in this dataset, together accounting for over half of the 340 companies. Telecommunications Services is by far the smallest sector (just 5 companies), which is worth keeping in mind when interpreting any sector-level average later in this notebook, since a mean computed over 5 companies is far less stable than one computed over 53.


In [ ]:
# Top 10 most common GICS Sub Industries (169 unique sub-industries exist, so only the largest are shown)
top_subindustries = df["GICS Sub Industry"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("Purples_r", len(top_subindustries))
ax.barh(top_subindustries.index[::-1], top_subindustries.values[::-1], color=colors[::-1])
ax.set_xlabel("Number of Companies")
ax.set_title("Top 10 GICS Sub Industries by Company Count")
plt.tight_layout()
plt.show()


**Insight:** No single sub-industry dominates the dataset: the largest, Oil & Gas Exploration & Production, has only 16 companies (under 5% of the total). This confirms that GICS Sub Industry is too fine-grained to use directly as a segmentation variable (169 distinct sub-industries for 340 companies), which is part of the motivation for using cluster analysis on the financial indicators instead of relying on the existing sector/sub-industry labels alone.


### Bivariate Analysis: Answering the Leading Questions


#### 1. Distribution of stock prices


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["Current Price"], bins=30, kde=True, color="#4C72B0", ax=axes[0])
axes[0].set_title("Distribution of Current Stock Price")
axes[0].set_xlabel("Current Price ($)")
sns.boxplot(x=df["Current Price"], color="#4C72B0", ax=axes[1])
axes[1].set_title("Current Stock Price: Boxplot")
axes[1].set_xlabel("Current Price ($)")
plt.tight_layout()
plt.show()


**Insight:** Stock prices are heavily right-skewed: most companies trade well under $150/share, with a long tail of high-priced stocks (a few above $500, one near $1,275). This is typical of a cross-section of large-cap NYSE stocks and confirms that price-based features should be scaled before clustering, since raw price alone isn't comparable across companies with different share structures.


#### 2. Average price change by sector


In [ ]:
sector_price_change = df.groupby("GICS Sector")["Price Change"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("Blues_r", len(sector_price_change))
ax.barh(sector_price_change.index[::-1], sector_price_change.values[::-1], color=colors[::-1])
ax.set_xlabel("Average Price Change (%, 13 weeks)")
ax.set_title("Average 13-Week Price Change by GICS Sector")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

sector_price_change


**Insight:** Health Care had the largest average 13-week price increase (+9.6%), followed by Consumer Staples and Information Technology. Energy was the only sector with a negative average price change (-10.2%), standing out clearly from the rest of the market over this period.


#### 3. Correlation between variables


In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={"label": "Correlation"}, ax=ax)
ax.set_title("Correlation Matrix of Financial Indicators")
plt.tight_layout()
plt.show()


**Insight:** Most financial indicators are weakly correlated with each other, which is good news for clustering (little redundant information). The strongest relationships are Net Income with Estimated Shares Outstanding (0.59) and Net Income with Earnings Per Share (0.56); larger, more profitable companies tend to have both more shares outstanding and higher EPS. Price Change and Volatility show little linear relationship with the size/profitability metrics, meaning market performance isn't simply explained by company size.


#### 4. Average cash ratio by sector


In [ ]:
sector_cash = df.groupby("GICS Sector")["Cash Ratio"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("Greens_r", len(sector_cash))
ax.barh(sector_cash.index[::-1], sector_cash.values[::-1], color=colors[::-1])
ax.set_xlabel("Average Cash Ratio")
ax.set_title("Average Cash Ratio by GICS Sector")
plt.tight_layout()
plt.show()

sector_cash


**Insight:** Information Technology companies carry by far the highest average cash ratio (~150), reflecting the sector's tendency to stockpile cash reserves. Utilities have the lowest (~14), consistent with a capital-intensive, debt-financed business model that doesn't need large cash buffers.


#### 5. Average P/E ratio by sector


In [ ]:
sector_pe = df.groupby("GICS Sector")["P/E Ratio"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("Purples_r", len(sector_pe))
ax.barh(sector_pe.index[::-1], sector_pe.values[::-1], color=colors[::-1])
ax.set_xlabel("Average P/E Ratio")
ax.set_title("Average P/E Ratio by GICS Sector")
plt.tight_layout()
plt.show()

sector_pe


**Insight:** Energy has, by a wide margin, the highest average P/E ratio (~73) despite having the worst average price performance, a sign that earnings in the sector were depressed over this period (a low denominator inflates P/E even when the stock itself is falling). Telecommunications Services and Financials have the lowest average P/E, suggesting the market was pricing those sectors more conservatively relative to earnings.


#### 6. Average volatility by sector


In [ ]:
sector_volatility = df.groupby("GICS Sector")["Volatility"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("Oranges_r", len(sector_volatility))
ax.barh(sector_volatility.index[::-1], sector_volatility.values[::-1], color=colors[::-1])
ax.set_xlabel("Average Volatility")
ax.set_title("Average Volatility by GICS Sector")
plt.tight_layout()
plt.show()

sector_volatility


**Insight:** Energy is both the worst-performing sector on average price change (see Q2) and the most volatile (~2.57), reinforcing that the sector's stocks carried the most risk over this period. Utilities is the calmest sector (~1.12), consistent with its reputation as a defensive, low-volatility holding. Pairing this with the price-change result gives a clearer risk-vs-return read than either chart alone: Energy's higher P/E and higher volatility both point to a sector where the market was pricing in more uncertainty, not just weaker recent performance.


## Data Preprocessing


* Duplicate value check
* Missing value treatment
* Outlier check
* Feature engineering (if needed)
* Any other preprocessing steps (if needed)


In [ ]:
# Duplicate and missing value checks were already done above: none found.

# Outlier check (IQR method) on the numeric features
feature_cols = ["Current Price", "Price Change", "Volatility", "ROE", "Cash Ratio",
                 "Net Cash Flow", "Net Income", "Earnings Per Share",
                 "Estimated Shares Outstanding", "P/E Ratio", "P/B Ratio"]

for c in feature_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[c] < lo) | (df[c] > hi)).sum()
    print(f"{c:32s} outliers={n_out:3d}")


**Outlier treatment decision:** Every numeric feature has a meaningful number of IQR outliers, but these reflect genuine differences in company scale and performance (mega-cap vs. small-cap, high-growth vs. distressed) rather than data-entry errors. Since the goal of this project is to *find* the natural groupings in the data, including the companies that are genuinely different from the pack, these outliers are retained rather than removed or capped. Their influence is instead managed by scaling all features before clustering, so no single high-magnitude column (e.g., Net Income in the billions) dominates the distance calculation.


In [ ]:
# Feature scaling: standardize every numeric feature to mean 0, std 1
# so no single feature (e.g. Net Income, in the billions) dominates the distance calculation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols, index=df.index)
X_scaled.describe().T[["mean", "std", "min", "max"]]


## EDA


* It is a good idea to explore the data once again after manipulating it.


In [ ]:
# Confirm scaling worked as expected: mean ~0, std ~1 for every feature
print("Feature means after scaling (should be ~0):")
print(X_scaled.mean().round(2))
print("\nFeature std devs after scaling (should be ~1):")
print(X_scaled.std().round(2))


**Insight:** After standardization, every feature has mean ~0 and standard deviation ~1, so no single financial indicator will dominate the distance calculations K-means and hierarchical clustering rely on. The scaled matrix (340 companies x 11 features) is now ready for clustering.


## K-means Clustering


### Finding the optimal number of clusters


In [ ]:
# Elbow method (distortion / inertia) across a range of k
distortions = {}
for k in range(2, 12):
    km = KMeans(n_clusters=k, random_state=1, n_init=10)
    km.fit(X_scaled)
    distortions[k] = km.inertia_

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(list(distortions.keys()), list(distortions.values()), marker="o", color="#4C72B0", label="Inertia")
ax.axvline(4, color="#C44E52", linestyle="--", label="Chosen k=4")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Distortion (Inertia)")
ax.set_title("Elbow Method for Optimal k (K-Means)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Silhouette score across the same range of k
sil_scores = {}
for k in range(2, 12):
    km = KMeans(n_clusters=k, random_state=1, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil_scores[k] = silhouette_score(X_scaled, labels)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(list(sil_scores.keys()), list(sil_scores.values()), marker="o", color="#4C72B0")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Average Silhouette Score")
ax.set_title("Silhouette Score by Number of Clusters (K-Means)")
plt.tight_layout()
plt.show()

sil_scores


**Choosing k:** The elbow in the distortion curve is fairly gradual, but flattens noticeably around k=4 to k=5. Silhouette scores are highest at k=3 (0.46) and k=4 (0.45), and drop sharply beyond k=6 as clusters start splitting into tiny, less meaningful groups. **k=4** is chosen as the final number of clusters: it sits at the elbow, has a strong silhouette score, and, as the cluster profiling below shows, produces four segments that are distinct and easy to act on from an investment standpoint (versus k=3, where two economically different groups got merged together).


In [ ]:
# Fit the final K-Means model with k=4
kmeans_final = KMeans(n_clusters=4, random_state=1, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_scaled)
df["KMeans_Cluster"] = kmeans_labels

print("Cluster sizes:")
print(df["KMeans_Cluster"].value_counts().sort_index())
print("\nSilhouette score (k=4):", silhouette_score(X_scaled, kmeans_labels))


In [ ]:
# Per-cluster silhouette visualizer for the chosen k=4 model
# (newer scikit-learn no longer sets _estimator_type, which yellowbrick's
# clustering visualizers check for; this one-line shim restores compatibility)
sil_model = KMeans(n_clusters=4, random_state=1, n_init=10)
sil_model._estimator_type = "clusterer"

fig, ax = plt.subplots(figsize=(10, 7))
sil_visualizer = SilhouetteVisualizer(sil_model, colors="yellowbrick", ax=ax)
sil_visualizer.fit(X_scaled)
sil_visualizer.finalize()
plt.tight_layout()
plt.show()


**Insight:** The four clusters are far from equally cohesive. Core Diversified Large-Caps (cluster 2, 278 companies) is very well separated: every member has a positive silhouette score, averaging 0.55. The three smaller, more specialized clusters are weaker: Mega-Cap Blue Chips (cluster 3, 11 companies) averages 0.23 with one company scoring slightly negative, Energy-Heavy Underperformers (cluster 1, 27 companies) averages just 0.05 with about a third of its members negative, and High-Growth/High-Valuation (cluster 0, 24 companies) actually averages *negative* (-0.14), with most of its members sitting closer to another cluster's centroid than their own. This makes sense given what these segments represent: they're defined by a small number of extreme companies (the highest-growth or highest-risk names) rather than a naturally dense group, so their boundaries with the broad Core Diversified cluster are blurry. It's a useful caveat for advisors: cluster assignments for the smaller, more extreme segments should be treated as directional rather than definitive for any single company near a boundary, while Core Diversified assignments can be trusted with high confidence.


### Cluster profiling


In [ ]:
# Mean of every feature within each K-Means cluster
cluster_profile = df.groupby("KMeans_Cluster")[feature_cols].mean().round(2)
cluster_profile


In [ ]:
# Boxplots of key differentiating features across the four K-Means clusters
key_feats = ["Price Change", "Volatility", "Cash Ratio", "P/E Ratio", "Net Income", "ROE"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
palette = sns.color_palette("Blues", 4)
for ax, feat in zip(axes.flat, key_feats):
    sns.boxplot(data=df, x="KMeans_Cluster", y=feat, hue="KMeans_Cluster", palette=palette, legend=False, ax=ax)
    ax.set_title(feat)
    ax.set_xlabel("Cluster")
fig.suptitle("K-Means Cluster Profiles (k=4)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Name each cluster based on its profile, then look at its sector composition
kmeans_cluster_names = {
    0: "High-Growth / High-Valuation",
    1: "Energy-Heavy Underperformers",
    2: "Core Diversified Large-Caps",
    3: "Mega-Cap Blue Chips",
}
df["KMeans_Cluster_Name"] = df["KMeans_Cluster"].map(kmeans_cluster_names)

sector_counts = pd.crosstab(df["KMeans_Cluster_Name"], df["GICS Sector"])
fig, ax = plt.subplots(figsize=(12, 6))
sector_counts.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_ylabel("Number of Companies")
ax.set_title("GICS Sector Composition by K-Means Cluster")
ax.legend(title="GICS Sector", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

df["KMeans_Cluster_Name"].value_counts()


**K-Means cluster profiles:**

* **Cluster 0: High-Growth / High-Valuation (24 companies):** High share prices (avg ~$243), strong 13-week price gains (+13.6%), high cash ratios, and rich P/E and P/B multiples. Concentrated in Health Care, Consumer Discretionary, and Information Technology (e.g., Amgen, McDonald's, Facebook). These are the stocks the market is paying a premium for on the expectation of continued growth.
* **Cluster 1: Energy-Heavy Underperformers (27 companies):** The only cluster with a negative average price change (-15.4%), dominated by Energy names (22 of 27), with an inflated average P/E driven by depressed earnings rather than strong prices. This cluster is a warning sign: a portfolio overweight here was likely hurt over this period.
* **Cluster 2: Core Diversified Large-Caps (278 companies):** The large "everything else" cluster, with moderate price change (+5.1%), moderate valuations, and no single dominant sector (Industrials, Financials, Consumer Discretionary, Health Care, and Real Estate all well represented). This is the market's broad middle.
* **Cluster 3: Mega-Cap Blue Chips (11 companies):** The smallest cluster by count but the largest by scale, with companies like JPMorgan Chase, Wells Fargo, and Gilead Sciences, by far the highest average Net Income and Estimated Shares Outstanding, low P/E, and negative average P/B (common for large financials with high leverage). These are the household-name giants.


## Hierarchical Clustering


### Comparing linkage methods


In [ ]:
# Compute linkage and cophenetic correlation for each of the four standard linkage methods
methods = ["single", "complete", "average", "ward"]
coph_scores = {}
linkages_dict = {}
for m in methods:
    Z = linkage(X_scaled, method=m)
    c, _ = cophenet(Z, pdist(X_scaled))
    coph_scores[m] = c
    linkages_dict[m] = Z

print("Cophenetic correlation by linkage method:")
for m, c in coph_scores.items():
    print(f"  {m:10s}: {c:.3f}")


In [ ]:
# Dendrogram for each linkage method (truncated to the last 30 merges for readability)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, m in zip(axes.flat, methods):
    dendrogram(linkages_dict[m], ax=ax, truncate_mode="lastp", p=30, no_labels=True)
    ax.set_title(f"{m.capitalize()} Linkage (cophenetic corr = {coph_scores[m]:.2f})")
    ax.set_xlabel("Cluster size (truncated)")
    ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()


In [ ]:
# Check resulting cluster sizes at k=4 for each linkage method
for m in methods:
    labels = fcluster(linkages_dict[m], t=4, criterion="maxclust")
    print(m, pd.Series(labels).value_counts().sort_index().to_dict())


**Linkage choice:** Average linkage has the highest cophenetic correlation (0.94), but at k=4 it, along with single and complete linkage, produces one giant cluster plus a handful of singleton outliers (a classic "chaining" effect on data with a few extreme companies). That structure isn't useful for portfolio segmentation. **Ward linkage** is chosen instead: its cophenetic correlation is lower (0.71) but it produces four balanced, well-separated clusters, which is the standard, expected trade-off when using Ward's variance-minimizing approach on data like this.


In [ ]:
# Dendrogram for the chosen Ward linkage, with the k=4 cut height marked
Z_ward = linkages_dict["ward"]
fig, ax = plt.subplots(figsize=(16, 7))
dendrogram(Z_ward, ax=ax, truncate_mode="lastp", p=40, leaf_rotation=90, leaf_font_size=8)
ax.set_title("Dendrogram: Ward Linkage (truncated to last 40 merges)")
ax.set_xlabel("Cluster size / sample index")
ax.set_ylabel("Distance")
ax.axhline(y=22, color="#C44E52", linestyle="--", label="Cut for 4 clusters")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Silhouette score across a range of k, cut from the Ward linkage
sil_scores_hac = {}
for k in range(2, 12):
    labels = fcluster(Z_ward, t=k, criterion="maxclust")
    sil_scores_hac[k] = silhouette_score(X_scaled, labels)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(list(sil_scores_hac.keys()), list(sil_scores_hac.values()), marker="o", color="#55A868")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Average Silhouette Score")
ax.set_title("Silhouette Score by Number of Clusters (Ward Linkage)")
plt.tight_layout()
plt.show()

sil_scores_hac


**Choosing k for hierarchical clustering:** Silhouette score under Ward linkage peaks at k=4 (0.465), matching both the K-means result and the natural cut visible in the dendrogram. k=4 is used for the final hierarchical solution, for direct comparability with K-means.


In [ ]:
# Fit the final hierarchical model: Ward linkage, k=4
hac_final = AgglomerativeClustering(n_clusters=4, linkage="ward")
hac_labels = hac_final.fit_predict(X_scaled)
df["HAC_Cluster"] = hac_labels

print("Cluster sizes:")
print(df["HAC_Cluster"].value_counts().sort_index())
print("\nSilhouette score (k=4, ward):", silhouette_score(X_scaled, hac_labels))


### Cluster profiling


In [ ]:
# Mean of every feature within each hierarchical (Ward) cluster
hac_profile = df.groupby("HAC_Cluster")[feature_cols].mean().round(2)
hac_profile


In [ ]:
# Top 3 GICS sectors within each hierarchical cluster
for c in sorted(df["HAC_Cluster"].unique()):
    print(f"Cluster {c} ({(df['HAC_Cluster']==c).sum()} stocks), top sectors:")
    print(df[df["HAC_Cluster"] == c]["GICS Sector"].value_counts().head(3).to_dict())
    print()


In [ ]:
# Same visual profiling as K-Means: boxplots of the key differentiating features by hierarchical cluster
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
palette_hac = sns.color_palette("Greens", 4)
for ax, feat in zip(axes.flat, key_feats):
    sns.boxplot(data=df, x="HAC_Cluster", y=feat, hue="HAC_Cluster", palette=palette_hac, legend=False, ax=ax)
    ax.set_title(feat)
    ax.set_xlabel("Cluster")
fig.suptitle("Hierarchical (Ward) Cluster Profiles (k=4)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Name each hierarchical cluster (based on its profile matching the same four segments found by K-Means),
# then look at its sector composition: the same view built for K-Means above, for a direct comparison.
hac_cluster_names = {
    0: "Energy-Heavy Underperformers",
    1: "High-Growth / High-Valuation",
    2: "Mega-Cap Blue Chips",
    3: "Core Diversified Large-Caps",
}
df["HAC_Cluster_Name"] = df["HAC_Cluster"].map(hac_cluster_names)

hac_sector_counts = pd.crosstab(df["HAC_Cluster_Name"], df["GICS Sector"])
fig, ax = plt.subplots(figsize=(12, 6))
hac_sector_counts.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_ylabel("Number of Companies")
ax.set_title("GICS Sector Composition by Hierarchical Cluster")
ax.legend(title="GICS Sector", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

df["HAC_Cluster_Name"].value_counts()


**Hierarchical cluster profiles (Ward, k=4):** The same four economic stories emerge as in K-means:

* **Cluster 0: Energy-Heavy Underperformers (29 companies):** The only hierarchical cluster with a negative average price change (-11.3%), dominated by Energy names (22 of 29) and carrying an elevated average P/E despite the price decline, the same depressed-earnings pattern seen in the K-Means equivalent.
* **Cluster 1: High-Growth / High-Valuation (15 companies):** The highest average share price (~$326), strong price gains (+10.6%), and the richest cash ratio and P/E multiples of the four clusters, a tighter, smaller version of the K-Means growth cluster (24 companies), with a few of the more moderate growth names instead landing in Cluster 3 here.
* **Cluster 2: Mega-Cap Blue Chips (11 companies):** By far the highest Net Income and Estimated Shares Outstanding, low P/E, and negative average P/B, essentially an exact match (10 of 11 companies overlap) with the K-Means Mega-Cap cluster.
* **Cluster 3: Core Diversified Large-Caps (285 companies):** The large, broad middle cluster with moderate valuations and no dominant sector (Industrials, Financials, and Consumer Discretionary lead but none dominates), again closely matching the K-Means core cluster (271 of 278 companies overlap).

Confirming that these groupings reflect real structure in the data rather than an artifact of one particular algorithm.


## K-means vs Hierarchical Clustering


In [ ]:
# Compare execution time for the two final models
import time

t0 = time.perf_counter()
KMeans(n_clusters=4, random_state=1, n_init=10).fit(X_scaled)
t1 = time.perf_counter()
AgglomerativeClustering(n_clusters=4, linkage="ward").fit(X_scaled)
t2 = time.perf_counter()

print(f"K-Means fit time:       {t1 - t0:.4f} sec")
print(f"Hierarchical fit time:  {t2 - t1:.4f} sec")


In [ ]:
# Cross-tab of cluster membership: how much do the two solutions agree?
agreement = pd.crosstab(df["KMeans_Cluster"], df["HAC_Cluster"])
agreement


**Comparison:**

* **Execution time:** Both methods are essentially instantaneous at this dataset size (340 rows); K-means was marginally slower in this run, but the difference is negligible. At much larger scale, K-means (which is roughly linear in the number of points) would pull ahead of hierarchical clustering (which is roughly quadratic).
* **Distinctness of clusters:** Both methods land on the same silhouette score (0.46-0.47) at k=4, and both isolate the same four economically distinct groups: Energy-heavy underperformers, high-growth/high-valuation names, a small mega-cap group, and one large core-diversified group.
* **Overlap between the two solutions:** The cross-tab above shows strong agreement: most companies assigned to a given K-means cluster land in the same corresponding hierarchical cluster, with only a modest number reshuffled at the boundaries between the "core diversified" group and the two smaller specialty groups.
* **Number of clusters chosen:** Both algorithms independently pointed to **k=4** as optimal by silhouette score, which is reassuring, since two different clustering approaches converged on the same structure.
* **Differences:** K-means gave a marginally cleaner separation for the mega-cap cluster (11 vs. companies split slightly differently under Ward), while hierarchical clustering's dendrogram gives an extra, useful diagnostic: it visually shows *how* clusters merge and at what distance, which K-means doesn't provide. For a one-off analysis like this, hierarchical clustering's dendrogram is a valuable exploratory tool; for repeated or larger-scale clustering (e.g., ongoing monthly re-segmentation), K-means would be the more practical, scalable choice.


## Actionable Insights and Recommendations


**Four clear investment segments emerged from the data**, consistent across two different clustering algorithms: Core Diversified Large-Caps (82% of stocks, the market's broad middle), High-Growth/High-Valuation names, Energy-Heavy Underperformers, and a small group of Mega-Cap Blue Chips. This structure gives Trade&Ahead's advisors a fast way to characterize any of the 340 stocks in the dataset without re-reading a full financial statement.

**The Energy-Heavy Underperformer cluster is the one to flag for risk management.** It's the only group with negative average price movement over the period (-15% in the hierarchical solution) and its elevated P/E ratios are driven by weak earnings rather than strong prices, a combination that can mislead an investor using P/E alone as a value signal. Portfolios overweight in Energy names from this cluster carried outsized downside risk over this period.

**The High-Growth/High-Valuation cluster commands rich multiples for a reason:** strong price momentum and healthy cash positions, but it also carries valuation risk: these are the stocks most exposed if growth expectations aren't met. They're better suited to investors with a higher risk tolerance and a longer time horizon than to conservative, income-focused portfolios.

**The Mega-Cap Blue Chip cluster, though only 11 companies, represents enormous scale** (the highest average Net Income and shares outstanding of any group) at comparatively low valuations (low P/E). These names offer scale and earnings stability and could anchor a more conservative portfolio, though their negative average P/B (common among highly leveraged financials) is worth a closer look company-by-company before relying on that metric alone.

**For diversification purposes, an advisor building a portfolio should draw from more than one cluster.** Because the clusters are built on financial characteristics rather than sector alone, picking one or two names from each of the four clusters, rather than several names from the same cluster, is a more direct way to reduce correlated risk than diversifying by GICS sector name alone, since (as seen in the sector-composition chart) some clusters cut across multiple sectors while others are dominated by a single one.

**Recommendation for next steps:** Re-run this clustering periodically (e.g., quarterly) as price, earnings, and cash-flow data update, since cluster membership, especially for borderline "Core Diversified" stocks, can shift as a company's growth trajectory or sector conditions change.
